# SemKey canonical Kaggle smoke
Set a user-controlled fork URL and immutable commit. Attach the private derived sharded dataset before running. This notebook performs validation and batch-1 smoke only; it does not train.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '65b251993827c0ff7412d78ce79694769387b7f0'
WORKTREE = '/kaggle/working/SemKey'
assert REPO_URL.startswith('https://github.com/') and 'REPLACE_' not in REPO_URL
assert len(COMMIT) == 40 and all(c in '0123456789abcdef' for c in COMMIT.lower())

In [ ]:
import glob, os, platform, subprocess, sys, torch
from kaggle_secrets import UserSecretsClient
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Add a private Kaggle Secret named GITHUB_TOKEN with read access to the repository'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("import os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual == COMMIT

In [ ]:
subprocess.run([sys.executable, os.path.join(WORKTREE, 'kaggle', 'smoke_input.py'), '--input-root', '/kaggle/input', '--batch-size', '1'], check=True)
manifest_paths = glob.glob('/kaggle/input/*/metadata/shard_manifest.json')
assert len(manifest_paths) == 1, manifest_paths
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
subprocess.run([sys.executable, os.path.join(WORKTREE, 'kaggle', 'smoke_semkey_sharded_loader.py'), '--dataset-root', dataset_root, '--phase', 'val'], check=True)
subprocess.run([sys.executable, '-m', 'py_compile', os.path.join(WORKTREE, 'data', 'datamodule.py'), os.path.join(WORKTREE, 'model', 'semkey_parallel.py')], check=True)
print('Kaggle canonical data and source smoke: PASS')